# 03 — Preprocessing dengan Apache Spark (Multi-Video)

Notebook ini membaca SEMUA komentar dari MongoDB (lintas video), menjalankan preprocessing dua jalur, dan menyimpan hasilnya.

- **Jalur A**: SVM + TF-IDF — cleaning agresif + stemming
- **Jalur B**: IndoBERT — cleaning minimal, morfologi terjaga

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from configs.config import Config
Config.paths.ensure_dirs()


In [2]:
from src.preprocess_spark import build_spark_session

spark = build_spark_session(
    app_name=Config.spark.APP_NAME,
    master=Config.spark.MASTER,
    mongo_package=Config.spark.MONGO_SPARK_PACKAGE if Config.spark.MONGO_CONNECTOR_JAR else None,
)

from src.notebook_helpers import check_spark_session
check_spark_session(spark)


2026-05-13 10:14:43 | INFO | preprocess_spark | SparkSession dibuat: YoutubeSentimentPipeline | local[*]

[OK] SparkSession aktif
  Versi Spark : 4.1.1
  App Name    : YoutubeSentimentPipeline
  Master      : local[*]


True

In [3]:
from src.mongo_utils import find_all_as_dicts, get_collection
from src.preprocess_spark import load_from_list_to_spark

collection_raw = get_collection(
    uri=Config.mongo.URI,
    db_name=Config.mongo.DB_NAME,
    collection_name=Config.mongo.COLLECTION_RAW,
)

# Baca SEMUA komentar dari semua video
print("Membaca semua komentar dari MongoDB...")
raw_records = find_all_as_dicts(
    collection_raw,
    projection={"_id": 0, "comment_id": 1, "video_id": 1, "text": 1},
)
print(f"Total komentar dibaca: {len(raw_records):,}")


2026-05-13 10:15:48 | INFO | mongo_utils | Koneksi MongoDB berhasil: mongodb://localhost:27017
Membaca semua komentar dari MongoDB...
Total komentar dibaca: 14,108


In [4]:
# Distribusi per video di Spark
df_raw = load_from_list_to_spark(
    spark, raw_records,
    select_columns=["comment_id", "video_id", "text"]
)

from pyspark.sql import functions as F
print("Distribusi komentar per video_id:")
df_raw.groupBy("video_id").count().orderBy(F.desc("count")).show(30, truncate=False)


Distribusi komentar per video_id:
+-----------+-----+
|video_id   |count|
+-----------+-----+
|TcNSqpBPle4|1250 |
|_8H_-uNHKL4|1164 |
|tQKnVXSL7nc|1073 |
|5btOJ_Wru2k|1070 |
|UEocTRWhtsE|1066 |
|g7Et92hQ_yE|1045 |
|uxDGtbE72jk|1039 |
|4yjXBLcjynU|1021 |
|LgaDFmaTpak|999  |
|HcwZx39bP80|958  |
|BMcbB3SlU1M|942  |
|u4zZ7gHRBaU|871  |
|UE2gk8vMgBQ|856  |
|UTvTHgjI_QI|754  |
+-----------+-----+



In [5]:
from src.preprocess_spark import preprocess_svm_path

print("\nJalur A: SVM + TF-IDF...")
df_svm = preprocess_svm_path(df_raw)

from src.notebook_helpers import dataframe_info
dataframe_info(df_svm, label="Jalur SVM Output")



Jalur A: SVM + TF-IDF...
2026-05-13 10:17:09 | INFO | preprocess_spark | Menjalankan Jalur A (SVM + TF-IDF)...


Py4JJavaError: An error occurred while calling o124.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 13 in stage 3.0 failed 1 times, most recent failure: Lost task 13.0 in stage 3.0 (TID 30) (LAPTOP-R2AKK3PS executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org.apache

In [ ]:
from src.preprocess_spark import preprocess_bert_path

print("\nJalur B: IndoBERT...")
df_bert = preprocess_bert_path(df_raw)
dataframe_info(df_bert, label="Jalur IndoBERT Output")


In [ ]:
from src.preprocess_spark import merge_preprocessed_paths

df_merged = merge_preprocessed_paths(df_raw, df_svm, df_bert)
print(f"Dataset gabungan: {df_merged.count():,} baris")
df_merged.show(5, truncate=60)


In [ ]:
from src.notebook_helpers import print_preprocessing_comparison

# Perbandingan preprocessing dari beberapa video berbeda
print("\n=== CONTOH PERBANDINGAN PREPROCESSING (LINTAS VIDEO) ===")
sample_rows = df_merged.limit(5).collect()
for row in sample_rows:
    print_preprocessing_comparison(
        original=row["text_original"],
        text_svm=row["text_svm"] or "",
        text_bert=row["text_bert"] or "",
    )


In [ ]:
# Simpan ke Parquet
parquet_path = str(Config.paths.DATA_PROCESSED / "merged_preprocessed")
df_merged.write.mode("overwrite").parquet(parquet_path)
print(f"Data tersimpan: {parquet_path}")

svm_path = str(Config.paths.DATA_PROCESSED / "svm")
bert_path = str(Config.paths.DATA_PROCESSED / "bert")
df_svm.write.mode("overwrite").parquet(svm_path)
df_bert.write.mode("overwrite").parquet(bert_path)
print(f"SVM Path  : {svm_path}")
print(f"BERT Path : {bert_path}")


## Ringkasan

Preprocessing dua jalur selesai untuk semua video:
- Dataset sudah mencakup komentar dari seluruh video yang berhasil di-ingest.
- Kolom `video_id` tetap ada sehingga data masih bisa dianalisis per video.
- Lanjutkan ke `04_export_labeling.ipynb`.